# **Crop Disease Detection**

## **Introduction**:

> #### *Problem Statement*: 
> > Few farmers are facing with issue with in the farm. There are lots of problems in the agricultural sector in the perspective of the farmer, but we can help them by integrating the AI Technology. Even after selecting and growing the crop in the field, few
crop diseases are not identified by the farmer which results to decrease in the crop yield. This is the main problem in the field so we are going to solve this issue by developing a model which can recognize the disease that is caused by the input of an image of the diseased plant. By
tackling this, we can increase the crop yield and maintain the crop production.

> #### *Overview of the project*:
> > This project involves building and loading the data, **Plant Village Dataset**, Exploratory Data Analysis, Model Building and Training.  
> #### *Goal of this project*:
> > The goal of this project is to build individual models for each vegetable as per the user wish and to observe the performance metrics of the model
> #### *Result*: 
> > With this we can hence use the model for any web application for crop managment systems or Crop Disease detection system

### Importing all the required libraries

In [ ]:
!pip install --no-deps --ignore-installed --quiet numpy==1.24.4 pandas==1.5.3 pyarrow==10.0.1 pytz==2023.3 pyyaml==6.0.1


In [ ]:


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Data handling
import pandas as pd
import numpy as np

# TensorFlow and Keras
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_addons as tfa
from tensorflow.python.client import device_lib
from tensorflow.keras.utils import image_dataset_from_directory, plot_model
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import TensorBoard
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras.preprocessing.image import ImageDataGenerator

# Visualizations
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

# Image and data preprocessing
from PIL import Image, ImageEnhance
import cv2
import random
import os
import math

# Sklearn tools
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer

# Utilities
from tqdm import tqdm

# Show available devices (e.g. GPU)
device_lib.list_local_devices()


## 1. Exploratory Data Analysis

### Defining a function to get the path for folder paths

In [ ]:
def get_path(plant_dir:str, dir_test:str):
    if dir_test == 'Test':
        return '/kaggle/input/plant-village-dataset-updated/' + plant_dir + '/Test'
    elif dir_test == 'Train':
        return '/kaggle/input/plant-village-dataset-updated/' + plant_dir + '/Train'
    elif dir_test == 'Valid':
        return '/kaggle/input/plant-village-dataset-updated/' + plant_dir + '/Val'
    
plant_dirs = os.listdir("/kaggle/input/plant-village-dataset-updated")
plant_dirs

### Printing the information about the Training set, the directories and the images

In [ ]:
img_dim = (256,256)
batch_size = 32
num_channels = 3
input_size = (batch_size, img_dim[0], img_dim[1], num_channels)
train_dataset = {}

print("-_-_-_-_-_-_-_-_-_-_Images & Classes for Training-_-_-_-_-_-_-_-_-_-_")
for plant in plant_dirs:
    print('>>> No of Images & Classes in "{}" directory'.format(plant))
    train_dataset[plant] = image_dataset_from_directory(get_path(plant, "Train"),
                                                        shuffle=True,
                                                        labels = 'inferred',
                                                        label_mode = 'int',
                                                        image_size = img_dim,
                                                        batch_size = batch_size)

### Printing the disease names of each plant

In [ ]:
classes  ={}
for plant in plant_dirs:
    print(">>> Classes in {} dataset :-".format(plant))
    classes[plant] = []
    for num, cat in enumerate(train_dataset[plant].class_names, start=1):
        classes[plant].append(cat)
        print(num, cat)
    print("\n")

### Plotting few random samples from each plant directory

In [ ]:
for plant in plant_dirs:
    print('>>>> Sample Images of "{}" dataset'.format(plant))
    plt.figure(figsize=(14,5))
    for image_batch, image_label in train_dataset[plant].take(1):
        for i in range(10):
            plt.subplot(2,5,i+1)
            plt.imshow(image_batch[i].numpy().astype('uint8'))
            plt.title(classes[plant][image_label[i]])
            plt.axis('off')
        plt.show()
    print("\n\n")

### Plotting the standardized images for randome plant leaf images

In [ ]:
def std_img(img):
    img_flat = img.reshape(-1,3)
    scaler = StandardScaler()
    img_std = scaler.fit_transform(img_flat)
    img_std = img_std.reshape(256,256,3)
    return img_std

tomato_img = []
tomato_label = []
for img,label in train_dataset["Tomato"].take(1):
    for i in range(5):
        tomato_img.append(img[i])
        tomato_label.append(classes["Tomato"][label[i]])
        
for i in range(5):        
    nik = np.array(tomato_img[i]).astype('uint8')
    img_std = std_img(nik)


    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(nik)
    plt.title("Tomato "+tomato_label[i])
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(img_std)
    plt.title('Standardized Image')
    plt.axis('off')
plt.show()

### Plotting the number of total images of each plant directory

In [ ]:
plants={}
dis = {}
for plant in os.listdir("/kaggle/input/plant-village-dataset-updated"):
    plant_count = 0
    path = f"/kaggle/input/plant-village-dataset-updated/{plant}/"
    for disease in os.listdir(path+"Train"):
        dis_count = 0
        for file in os.listdir(path+f"Train/{disease}"):
            plant_count+=1
            dis_count +=1
        dis[plant+" "+disease] = dis_count
    for disease in os.listdir(path+"Test"):
        for file in os.listdir(path+f"Test/{disease}"):
            plant_count+=1    
    for disease in os.listdir(path+"Val"):
        for file in os.listdir(path+f"Val/{disease}"):
            plant_count+=1
    plants[plant] = plant_count

  
categories = list(plants.keys())
counts = list(plants.values())

# Plotting the bar plot
plt.figure(figsize=(12, 6))
plt.bar(categories, counts, color='skyblue', edgecolor='black')
plt.xlabel('Vegetables', fontsize=14)
plt.ylabel('Counts', fontsize= 14)
plt.title('Count of Each Vegetable', fontsize=20)
plt.xticks(ha='center')  # Rotate x-axis labels for better readability
plt.show()

### Plotting the number of diseases per each plant

In [ ]:
plts = os.listdir("/kaggle/input/plant-village-dataset-updated")
x = list(dis.keys())
y = list(dis.values())
colors = plt.cm.viridis(np.linspace(0, 1, 6)) 
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes[0, 0].bar(x[:6], y[:6], color=colors[:6], label=x[:6])
axes[0, 1].bar(x[6:10], y[6:10], color=colors[:4], label=x[6:10])
axes[0, 2].bar(x[10:12], y[10:12], color=colors[-2:], label=x[10:12])
axes[1, 0].bar(x[12:14], y[12:14], color=colors[-2:], label=x[12:14])
axes[1, 1].bar(x[14:18], y[14:18], color=colors[-4:], label=x[14:18])
axes[1, 2].bar(x[18:20], y[18:20], color=colors[-2:], label=x[18:20])
axes[2, 0].bar(x[20:24], y[20:24], color=colors[-4:], label=x[20:24])
axes[2, 1].bar(x[24:26], y[24:26], color=colors[-2:], label=x[24:26])
axes[2, 2].bar(x[26:29], y[26:29], color=colors[-3:], label=x[26:29])
for i in range(3):
    for j in range(3):
        idx = i * 3 + j
        axes[i, j].set_xticks([])
        axes[i, j].set_xlabel('Diseases', fontsize=13)
        axes[i, j].set_ylabel('Count', fontsize=13)
        axes[i, j].set_title(plts[idx], fontsize=15)
        axes[i, j].legend()

plt.show()

### Plotting the 32X32 patch format of a image

In [ ]:
image_size = 224 
img_height, img_width = 512, 512
patch_size = 32
num_patches = (image_size // patch_size) ** 2
class Patches(layers.Layer):
    
    def __init__(self, patch_size):
        super(Patches, self).__init__()
        super(Patches, self).__init__()
        self.patch_size = patch_size
    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches
    
    
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
plt.figure(figsize=(4, 4))

target = "/kaggle/input/plant-village-dataset-updated/Apple/Train/Black Rot"
random_num = random.choice(os.listdir(target))
image = mpimg.imread("/kaggle/input/plant-village-dataset-updated/Apple/Train/Black Rot/" + random_num)
plt.imshow(image.astype("uint8"))
plt.axis("off")
resized_image = tf.image.resize(
    tf.convert_to_tensor([image]), size=(image_size, image_size)
)
patches = Patches(patch_size)(resized_image)
print(f"Image size: {image_size} X {image_size}")
print(f"Patch size: {patch_size} X {patch_size}")
print(f"Patches per image: {patches.shape[1]}")
print(f"Elements per patch: {patches.shape[-1]}")
n = int(np.sqrt(patches.shape[1]))
plt.figure(figsize=(4, 4))
for i, patch in enumerate(patches[0]):
    ax = plt.subplot(n, n, i + 1)
    patch_img = tf.reshape(patch, (patch_size, patch_size, 3))
    plt.imshow(patch_img.numpy().astype("uint8"))
    plt.axis("off")


# **Model Development**

### Enter the plant name for which you want to develop the model, as the variable

In [ ]:
plant = "Strawberry"

### Declaring the paths for the training, testing, validation path, labels for the given plant diseases and the output length

In [ ]:
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
out_len = len(out_labels)

### Declaring the batch size of 64 and the image size of 224X224 pixels


In [ ]:
batch_size = 64
img_height = 224

### Declare the ImageDataGenerator for the train_datagen and test_datagen, val_datagen. For train_datagen the images are augumented and for all the three datagenerators the pixels are scaled

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255.,rotation_range=20,shear_range=0.15,horizontal_flip=True,)
val_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

### Declaring the train, test and valid sets for the input to the model, making every image to be of the size 224X224 pixels


In [ ]:
train_set = train_datagen.flow_from_directory(train_path,target_size = (224,224),batch_size = 64,shuffle = True,class_mode = 'categorical')
val_set = val_datagen.flow_from_directory(val_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')
test_set = val_datagen.flow_from_directory(test_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')

### Loading the pretrained VIT model with RESNET50 as backbone with 32 patch size and pretrained on ImageNet-21k dataset. Building a classifier head over the pretrained VIT Model with out_len as the output shape which is equal to the number of disease of plant

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# Set proper input size
image_size = 224  # Can be 512 if you want high resolution
INPUT_SHAPE = (image_size, image_size, 3)
NUM_CLASSES = out_len  # Replace with actual number of classes

# Load the pretrained ResNet50 model without the top classification layers
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=INPUT_SHAPE,
    pooling='avg'  # GlobalAveragePooling2D is applied automatically
)

# Freeze the base model
base_model.trainable = False

# Build the full model
resnet_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=INPUT_SHAPE),
    layers.Lambda(preprocess_input),  # ResNet-specific preprocessing
    base_model,
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')
])

# Compile the model
resnet_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)


fe_L2 = hub.KerasLayer("https://tfhub.dev/sayakpaul/vit_r50_l32_fe/1",input_shape = (224,224,3),trainable = False,name = "Pre_Trained_")
VIT = tf.keras.Sequential([
    fe_L2,
    layers.Dense(128,activation = "relu"),
    layers.Dropout(0.5),
    layers.Dense(out_len, activation = "softmax", name = "output_layer")
])

VIT.compile(loss = "categorical_crossentropy",optimizer = tf.keras.optimizers.Adam(learning_rate = 0.0001),metrics = ["accuracy"])

### Trianing the model for 10 epochs

In [ ]:
epochs = 10

r = resnet_model.fit(
    train_set,
    epochs=epochs,
    validation_data=val_set
)
 

epochs = 10
r=VIT.fit(train_set,epochs = epochs,validation_data = val_set,steps_per_epoch=len(train_set),validation_steps = len(val_set))

### Saving the model with plant name

In [ ]:
model_name = plant + "_model"
resnet_model.save(model_name)

### Predicting the test_set for the model

In [ ]:
import numpy as np


Y_pred = resnet_model.predict(test_set, steps=len(test_set))
y_pred = np.argmax(Y_pred, axis=1)


### Plotting the confusion matrix for given y_pred and y_true

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib import pyplot as plt
cf = confusion_matrix(test_set.classes, y_pred)
list = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
plt.figure(figsize=(8, 6))
sns.heatmap(cf, annot=True, fmt='d', cmap='Blues',xticklabels=out_labels,yticklabels=out_labels)
plt.title(f'Confusion Matrix of {plant} with Accuracy : {accuracy_score(test_set.classes, y_pred) * 100:.2f}%')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt_name = plant+"_CF.png"
plt.savefig(plt_name)
plt.show()

### Printing the classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print('-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_>>>>Classification Report<<<<-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_')
print(classification_report(test_set.classes, y_pred, target_names=out_labels))

### Plotting the model Accuracy and Loss Graphs

In [ ]:
epochs = [i for i in range(10)]
fig , ax = plt.subplots(1,2)
train_acc = r.history['accuracy']
train_loss = r.history['loss']
val_acc = r.history['val_accuracy']
val_loss = r.history['val_loss']
fig.set_size_inches(16,9)

ax[0].plot(epochs , train_acc , 'go-' , label = 'Training Accuracy')
ax[0].plot(epochs , val_acc , 'ro-' , label = 'Testing Accuracy')
ax[0].set_title('Training & Validation Accuracy')
ax[0].legend()
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Accuracy")

ax[1].plot(epochs , train_loss , 'g-o' , label = 'Training Loss')
ax[1].plot(epochs , val_loss , 'r-o' , label = 'Testing Loss')
ax[1].set_title('Testing Accuracy & Loss')
ax[1].legend()
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Loss")

plt.show()

In [ ]:
#Federated learning environmnet 
#1 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = "Strawberry"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 1
NUM_ROUNDS = 2
EPOCHS_PER_CLIENT = 2

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


In [ ]:
#Federated learning environmnet 
#3 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = "Strawberry"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 3
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


# **Model Development**

**Enter the plant name for which you want to develop the model, as the variable**

In [ ]:
plant = "Apple"

In [ ]:
import os
from keras.preprocessing.image import ImageDataGenerator
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
out_len = len(out_labels)



In [ ]:
batch_size = 64
img_height = 224

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255.,rotation_range=20,shear_range=0.15,horizontal_flip=True,)
val_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

In [ ]:
train_set = train_datagen.flow_from_directory(train_path,target_size = (224,224),batch_size = 64,shuffle = True,class_mode = 'categorical')
val_set = val_datagen.flow_from_directory(val_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')
test_set = val_datagen.flow_from_directory(test_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# Set proper input size
image_size = 224  # Can be 512 if you want high resolution
INPUT_SHAPE = (image_size, image_size, 3)
NUM_CLASSES = out_len  # Replace with actual number of classes

# Load the pretrained ResNet50 model without the top classification layers
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=INPUT_SHAPE,
    pooling='avg'  # GlobalAveragePooling2D is applied automatically
)

# Freeze the base model
base_model.trainable = False

# Build the full model
resnet_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=INPUT_SHAPE),
    layers.Lambda(preprocess_input),  # ResNet-specific preprocessing
    base_model,
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')
])

# Compile the model
resnet_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)


In [ ]:
epochs = 10

r = resnet_model.fit(
    train_set,
    epochs=epochs,
    validation_data=val_set
)


In [ ]:
import numpy as np
model_name = plant + "_model"
resnet_model.save(model_name)


Y_pred = resnet_model.predict(test_set, steps=len(test_set))
y_pred = np.argmax(Y_pred, axis=1)


In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib import pyplot as plt
cf = confusion_matrix(test_set.classes, y_pred)
list = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
plt.figure(figsize=(8, 6))
sns.heatmap(cf, annot=True, fmt='d', cmap='Blues',xticklabels=out_labels,yticklabels=out_labels)
plt.title(f'Confusion Matrix of {plant} with Accuracy : {accuracy_score(test_set.classes, y_pred) * 100:.2f}%')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt_name = plant+"_CF.png"
plt.savefig(plt_name)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print('-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_>>>>Classification Report<<<<-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_')
print(classification_report(test_set.classes, y_pred, target_names=out_labels))

In [ ]:
from matplotlib import pyplot as plt
epochs = [i for i in range(10)]
fig , ax = plt.subplots(1,2)
train_acc = r.history['accuracy']
train_loss = r.history['loss']
val_acc = r.history['val_accuracy']
val_loss = r.history['val_loss']
fig.set_size_inches(16,9)

ax[0].plot(epochs , train_acc , 'go-' , label = 'Training Accuracy')
ax[0].plot(epochs , val_acc , 'ro-' , label = 'Testing Accuracy')
ax[0].set_title('Training & Validation Accuracy')
ax[0].legend()
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Accuracy")

ax[1].plot(epochs , train_loss , 'g-o' , label = 'Training Loss')
ax[1].plot(epochs , val_loss , 'r-o' , label = 'Testing Loss')
ax[1].set_title('Testing Accuracy & Loss')
ax[1].legend()
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Loss")

plt.show()

In [ ]:
#Federated learning environmnet 
#1 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = 'Apple'
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 1
NUM_ROUNDS = 2
EPOCHS_PER_CLIENT = 2

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


In [ ]:
#Federated learning environmnet 
#3 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = "Apple"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 3
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


In [ ]:
'''#Federated learning environmnet 
#6 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = "Apple"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 6
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')'''


# **Model Development**

In [ ]:
plant="Bell Pepper"

In [ ]:
import os
from keras.preprocessing.image import ImageDataGenerator
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
out_len = len(out_labels)

In [ ]:
batch_size = 64
img_height = 224

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255.,rotation_range=20,shear_range=0.15,horizontal_flip=True,)
val_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

In [ ]:
train_set = train_datagen.flow_from_directory(train_path,target_size = (224,224),batch_size = 64,shuffle = True,class_mode = 'categorical')
val_set = val_datagen.flow_from_directory(val_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')
test_set = val_datagen.flow_from_directory(test_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# Set proper input size
image_size = 224  # Can be 512 if you want high resolution
INPUT_SHAPE = (image_size, image_size, 3)
NUM_CLASSES = out_len  # Replace with actual number of classes

# Load the pretrained ResNet50 model without the top classification layers
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=INPUT_SHAPE,
    pooling='avg'  # GlobalAveragePooling2D is applied automatically
)

# Freeze the base model
base_model.trainable = False

# Build the full model
resnet_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=INPUT_SHAPE),
    layers.Lambda(preprocess_input),  # ResNet-specific preprocessing
    base_model,
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')
])
# Compile the model
resnet_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)


In [ ]:

epochs = 10

r = resnet_model.fit(
    train_set,
    epochs=epochs,
    validation_data=val_set
)


In [ ]:
import numpy as np
model_name = plant + "_model"
resnet_model.save(model_name)


Y_pred = resnet_model.predict(test_set, steps=len(test_set))
y_pred = np.argmax(Y_pred, axis=1)

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib import pyplot as plt
cf = confusion_matrix(test_set.classes, y_pred)
list = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
plt.figure(figsize=(8, 6))
sns.heatmap(cf, annot=True, fmt='d', cmap='Blues',xticklabels=out_labels,yticklabels=out_labels)
plt.title(f'Confusion Matrix of {plant} with Accuracy : {accuracy_score(test_set.classes, y_pred) * 100:.2f}%')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt_name = plant+"_CF.png"
plt.savefig(plt_name)
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print('-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_>>>>Classification Report<<<<-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_')
print(classification_report(test_set.classes, y_pred, target_names=out_labels))





In [ ]:



from matplotlib import pyplot as plt
epochs = [i for i in range(10)]
fig , ax = plt.subplots(1,2)
train_acc = r.history['accuracy']
train_loss = r.history['loss']
val_acc = r.history['val_accuracy']
val_loss = r.history['val_loss']
fig.set_size_inches(16,9)

ax[0].plot(epochs , train_acc , 'go-' , label = 'Training Accuracy')
ax[0].plot(epochs , val_acc , 'ro-' , label = 'Testing Accuracy')
ax[0].set_title('Training & Validation Accuracy')
ax[0].legend()
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Accuracy")

ax[1].plot(epochs , train_loss , 'g-o' , label = 'Training Loss')
ax[1].plot(epochs , val_loss , 'r-o' , label = 'Testing Loss')
ax[1].set_title('Testing Accuracy & Loss')
ax[1].legend()
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Loss")

plt.show()



In [ ]:
#Federated learning environmnet 
#1 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = 'Bell Pepper'
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 1
NUM_ROUNDS = 2
EPOCHS_PER_CLIENT = 2

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


In [ ]:
#Federated learning environmnet 
#3 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = "Bell Pepper"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 3
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


In [ ]:
'''#Federated learning environmnet 
#6 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = "Bell Pepper"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 6
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')'''


# **Model Development**

In [ ]:
plant="Peach"

In [ ]:
import os
from keras.preprocessing.image import ImageDataGenerator
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
out_len = len(out_labels)


In [ ]:
batch_size = 64
img_height = 224


In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255.,rotation_range=20,shear_range=0.15,horizontal_flip=True,)
val_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

In [ ]:
train_set = train_datagen.flow_from_directory(train_path,target_size = (224,224),batch_size = 64,shuffle = True,class_mode = 'categorical')
val_set = val_datagen.flow_from_directory(val_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')
test_set = val_datagen.flow_from_directory(test_path,target_size = (224, 224),batch_size = 64,shuffle = False,class_mode = 'categorical')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
# Set proper input size
image_size = 224  # Can be 512 if you want high resolution
INPUT_SHAPE = (image_size, image_size, 3)
NUM_CLASSES = out_len  # Replace with actual number of classes

# Load the pretrained ResNet50 model without the top classification layers
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=INPUT_SHAPE,
    pooling='avg'  # GlobalAveragePooling2D is applied automatically
)

# Freeze the base model
base_model.trainable = False

# Build the full model
resnet_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=INPUT_SHAPE),
    layers.Lambda(preprocess_input),  # ResNet-specific preprocessing
    base_model,
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')
])

# Compile the model
resnet_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)




In [ ]:
epochs = 10

r = resnet_model.fit(
    train_set,
    epochs=epochs,
    validation_data=val_set
)

In [ ]:

import numpy as np
model_name = plant + "_model"
resnet_model.save(model_name)


Y_pred = resnet_model.predict(test_set, steps=len(test_set))
y_pred = np.argmax(Y_pred, axis=1)

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib import pyplot as plt
cf = confusion_matrix(test_set.classes, y_pred)
list = os.listdir(f"/kaggle/input/plant-village-dataset-updated/{plant}/Train")
plt.figure(figsize=(8, 6))
sns.heatmap(cf, annot=True, fmt='d', cmap='Blues',xticklabels=out_labels,yticklabels=out_labels)
plt.title(f'Confusion Matrix of {plant} with Accuracy : {accuracy_score(test_set.classes, y_pred) * 100:.2f}%')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt_name = plant+"_CF.png"
plt.savefig(plt_name)
plt.show()

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print('-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_>>>>Classification Report<<<<-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_')
print(classification_report(test_set.classes, y_pred, target_names=out_labels))


In [ ]:

from matplotlib import pyplot as plt
epochs = [i for i in range(10)]
fig , ax = plt.subplots(1,2)
train_acc = r.history['accuracy']
train_loss = r.history['loss']
val_acc = r.history['val_accuracy']
val_loss = r.history['val_loss']
fig.set_size_inches(16,9)

ax[0].plot(epochs , train_acc , 'go-' , label = 'Training Accuracy')
ax[0].plot(epochs , val_acc , 'ro-' , label = 'Testing Accuracy')
ax[0].set_title('Training & Validation Accuracy')
ax[0].legend()
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Accuracy")

ax[1].plot(epochs , train_loss , 'g-o' , label = 'Training Loss')
ax[1].plot(epochs , val_loss , 'r-o' , label = 'Testing Loss')
ax[1].set_title('Testing Accuracy & Loss')
ax[1].legend()
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Loss")

plt.show()



In [ ]:
#Federated learning environmnet 
#1 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = 'Peach'
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 1
NUM_ROUNDS = 2
EPOCHS_PER_CLIENT = 2

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


In [ ]:
#Federated learning environmnet 
#3 clients
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
import os

# Paths and parameters
plant = "Peach"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 3
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generators
datagen = ImageDataGenerator(rescale=1./255.)

# Create the datasets
train_set = datagen.flow_from_directory(train_path, target_size=(224, 224), batch_size=batch_size, shuffle=True, class_mode='categorical')
val_set = datagen.flow_from_directory(val_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')
test_set = datagen.flow_from_directory(test_path, target_size=(224, 224), batch_size=batch_size, shuffle=False, class_mode='categorical')

# Function to create client-specific data generators
def create_client_data_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_gen = datagen.flow_from_directory(
            train_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=True,
            class_mode='categorical'
        )
        client_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_gen.samples = len(client_gen.filenames)
        client_generators.append(client_gen)
    return client_generators

# Function to create client-specific validation data generators
def create_client_val_generators(num_clients, dataset, batch_size):
    total_batches = len(dataset)
    client_val_generators = []
    for i in range(num_clients):
        start_idx = i * total_batches // num_clients
        end_idx = (i + 1) * total_batches // num_clients
        client_val_gen = datagen.flow_from_directory(
            val_path,
            target_size=(224, 224),
            batch_size=batch_size,
            shuffle=False,
            class_mode='categorical'
        )
        client_val_gen.filenames = dataset.filenames[start_idx:end_idx]
        client_val_gen.samples = len(client_val_gen.filenames)
        client_val_generators.append(client_val_gen)
    return client_val_generators

# Create client data generators
client_train_generators = create_client_data_generators(NUM_CLIENTS, train_set, batch_size)
client_val_generators = create_client_val_generators(NUM_CLIENTS, val_set, batch_size)

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    base_model.trainable = False  # Freeze ResNet base

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        layers.Lambda(preprocess_input),  # Preprocessing specific to ResNet
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax', name='output_layer')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model


# Federated training loop
global_model = create_model()
for round_num in range(NUM_ROUNDS):
    client_models = []
    client_accuracies = []
    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT)

        # Evaluate on client-specific validation data
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index])
        client_accuracies.append(val_acc)
        print(f'Client {client_index + 1}: Validation Accuracy = {val_acc:.4f}')

        client_models.append(client_model)

    # FedAvg aggregation
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set)
    print(f'Round {round_num + 1}: Validation Loss = {val_loss:.4f}, Validation Accuracy = {val_acc:.4f}')

# Final evaluation on the test set
test_loss, test_acc = global_model.evaluate(test_set)
print(f'Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}')


In [ ]:
# Federated learning with FedAdam optimizer and ResNet50 model
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers

# Paths and parameters
plant = "Peach"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 3
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generator
datagen = ImageDataGenerator(rescale=1./255.)

def get_data_generator(path, shuffle=True):
    return datagen.flow_from_directory(
        
        path,
        target_size=(img_height, img_height),
        batch_size=batch_size,
        shuffle=shuffle,
        class_mode='categorical'  # One-hot labels
    )

train_set = get_data_generator(train_path)
val_set = get_data_generator(val_path, shuffle=False)
test_set = get_data_generator(test_path, shuffle=False)

# Client data generators

def create_client_generators(base_path, num_clients):
    base_gen = get_data_generator(base_path)
    total_samples = len(base_gen.filenames)
    client_generators = []

    for i in range(num_clients):
        start = i * total_samples // num_clients
        end = (i + 1) * total_samples // num_clients
        subset_gen = get_data_generator(base_path)
        subset_gen.filenames = base_gen.filenames[start:end]
        subset_gen.samples = len(subset_gen.filenames)
        client_generators.append(subset_gen)

    return client_generators

client_train_generators = create_client_generators(train_path, NUM_CLIENTS)
client_val_generators = create_client_generators(val_path, NUM_CLIENTS)

# Model definition

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(img_height, img_height, 3),
        pooling='avg'
    )
    base_model.trainable = False

    model = tf.keras.Sequential([
        layers.Input(shape=(img_height, img_height, 3)),
        layers.Lambda(preprocess_input),
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(1e-4),
        metrics=['accuracy']
    )
    return model

# Federated training loop with FedAdam
global_model = create_model()

for round_num in range(NUM_ROUNDS):
    client_weights = []
    client_accuracies = []

    for client_idx in range(NUM_CLIENTS):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_train_generators[client_idx], epochs=EPOCHS_PER_CLIENT, verbose=0)
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_idx], verbose=0)
        print(f"Client {client_idx + 1} - Validation Accuracy: {val_acc:.4f}")
        client_weights.append(client_model.get_weights())
        client_accuracies.append(val_acc)

    # FedAdam-style aggregation
    new_weights = []
    for weights in zip(*client_weights):
        new_weights.append(np.mean(weights, axis=0))

    global_model.set_weights(new_weights)

    val_loss, val_acc = global_model.evaluate(val_set, verbose=0)
    print(f"Round {round_num + 1} - Global Validation Accuracy: {val_acc:.4f}")

# Final evaluation
test_loss, test_acc = global_model.evaluate(test_set, verbose=0)
print(f"Final Test Accuracy: {test_acc:.4f}")


In [ ]:
# Federated learning with FedAdam optimizer and ResNet50 model
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers

# Paths and parameters
plant = "Bell Pepper"
train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"
out_labels = os.listdir(train_path)
out_len = len(out_labels)
batch_size = 64
img_height = 224
NUM_CLIENTS = 3
NUM_ROUNDS = 3
EPOCHS_PER_CLIENT = 6

# Data generator
datagen = ImageDataGenerator(rescale=1./255.)

def get_data_generator(path, shuffle=True):
    return datagen.flow_from_directory(
        
        path,
        target_size=(img_height, img_height),
        batch_size=batch_size,
        shuffle=shuffle,
        class_mode='categorical'  # One-hot labels
    )

train_set = get_data_generator(train_path)
val_set = get_data_generator(val_path, shuffle=False)
test_set = get_data_generator(test_path, shuffle=False)

# Client data generators

def create_client_generators(base_path, num_clients):
    base_gen = get_data_generator(base_path)
    total_samples = len(base_gen.filenames)
    client_generators = []

    for i in range(num_clients):
        start = i * total_samples // num_clients
        end = (i + 1) * total_samples // num_clients
        subset_gen = get_data_generator(base_path)
        subset_gen.filenames = base_gen.filenames[start:end]
        subset_gen.samples = len(subset_gen.filenames)
        client_generators.append(subset_gen)

    return client_generators

client_train_generators = create_client_generators(train_path, NUM_CLIENTS)
client_val_generators = create_client_generators(val_path, NUM_CLIENTS)

# Model definition

def create_model():
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(img_height, img_height, 3),
        pooling='avg'
    )
    base_model.trainable = False

    model = tf.keras.Sequential([
        layers.Input(shape=(img_height, img_height, 3)),
        layers.Lambda(preprocess_input),
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(out_len, activation='softmax')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(1e-4),
        metrics=['accuracy']
    )
    return model

# Federated training loop with FedAdam
global_model = create_model()

for round_num in range(NUM_ROUNDS):
    client_weights = []
    client_accuracies = []

    for client_idx in range(NUM_CLIENTS):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_train_generators[client_idx], epochs=EPOCHS_PER_CLIENT, verbose=0)
        val_loss, val_acc = client_model.evaluate(client_val_generators[client_idx], verbose=0)
        print(f"Client {client_idx + 1} - Validation Accuracy: {val_acc:.4f}")
        client_weights.append(client_model.get_weights())
        client_accuracies.append(val_acc)

    # FedAdam-style aggregation
    new_weights = []
    for weights in zip(*client_weights):
        new_weights.append(np.mean(weights, axis=0))

    global_model.set_weights(new_weights)

    val_loss, val_acc = global_model.evaluate(val_set, verbose=0)
    print(f"Round {round_num + 1} - Global Validation Accuracy: {val_acc:.4f}")

# Final evaluation
test_loss, test_acc = global_model.evaluate(test_set, verbose=0)
print(f"Final Test Accuracy: {test_acc:.4f}")


In [ ]:
# Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

train_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Train"
val_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Val"
test_path = f"/kaggle/input/plant-village-dataset-updated/{plant}/Test"


batch_size  = 32
img_height  = 224
img_width   = 224

# Create an image data generator (rescale pixel values)
datagen = ImageDataGenerator(rescale=1./255)

# Load the full training, validation, and test sets from directories
full_train_gen = datagen.flow_from_directory(
    train_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
    class_mode='categorical'
)

full_val_gen = datagen.flow_from_directory(
    val_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False,
    class_mode='categorical'
)

test_gen = datagen.flow_from_directory(
    test_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False,
    class_mode='categorical'
)

# Number of classes and total samples
num_classes = len(full_train_gen.class_indices)
total_train_samples = full_train_gen.samples
print(f"Total training samples: {total_train_samples}, Classes: {full_train_gen.class_indices}")

# Partition the training data among clients
NUM_CLIENTS = 3
client_train_gens = []
for i in range(NUM_CLIENTS):
    # Compute index range for this client
    start = int(i * total_train_samples / NUM_CLIENTS)
    end   = int((i+1) * total_train_samples / NUM_CLIENTS) if i < NUM_CLIENTS - 1 else total_train_samples
    
    # Create a new generator for this client (same source directory)
    client_gen = datagen.flow_from_directory(
        train_path,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        shuffle=True,
        class_mode='categorical'
    )
    # Slice filenames and classes for this client
    client_gen.filenames = full_train_gen.filenames[start:end]
    client_gen.classes   = full_train_gen.classes[start:end]
    client_gen.samples   = len(client_gen.filenames)
    
    client_train_gens.append(client_gen)
    print(f"Client {i+1}: {client_gen.samples} training samples.")

# (Optional) Partition the validation set similarly for client-specific evaluation
total_val_samples = full_val_gen.samples
client_val_gens = []
for i in range(NUM_CLIENTS):
    start = int(i * total_val_samples / NUM_CLIENTS)
    end   = int((i+1) * total_val_samples / NUM_CLIENTS) if i < NUM_CLIENTS - 1 else total_val_samples
    
    client_val = datagen.flow_from_directory(
        val_path,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        shuffle=False,
        class_mode='categorical'
    )
    client_val.filenames = full_val_gen.filenames[start:end]
    client_val.classes   = full_val_gen.classes[start:end]
    client_val.samples   = len(client_val.filenames)
    
    client_val_gens.append(client_val)
    print(f"Client {i+1}: {client_val.samples} validation samples.")


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, Sequential, optimizers

def create_model():
    """
    Build a CNN model with a frozen ResNet50 base and custom head.
    Matches the architecture from the notebook:contentReference[oaicite:6]{index=6}.
    """
    base_model = ResNet50(include_top=False, weights='imagenet', pooling='avg',
                          input_shape=(img_height, img_width, 3))
    base_model.trainable = False  # Freeze ResNet base
    
    model = Sequential([
        layers.Input(shape=(img_height, img_width, 3)),
        layers.Lambda(lambda x: x, name='identity'),  # No-op, since we already rescaled
        base_model,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(
        loss='categorical_crossentropy',
        optimizer=optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )
    return model


In [ ]:
def fed_adam_aggregate(global_weights, client_weights, client_sizes, m, v,
                       server_lr=0.1, beta1=0.9, beta2=0.99, eps=1e-9):
    """
    Aggregate client updates using FedAdam (with momentum m and variance v).
    """
    total_samples = sum(client_sizes)

    # Compute weighted average of client weight differences
    weighted_delta = [np.zeros_like(w) for w in global_weights]
    for cw, n in zip(client_weights, client_sizes):
        for idx, (gw_layer, cw_layer) in enumerate(zip(global_weights, cw)):
            delta = cw_layer - gw_layer
            weighted_delta[idx] += (n / total_samples) * delta

    # Update momentum and variance
    new_m = []
    new_v = []
    for idx, delta_layer in enumerate(weighted_delta):
        m_prev = m[idx]
        v_prev = v[idx]
        m_curr = beta1 * m_prev + (1 - beta1) * delta_layer
        v_curr = beta2 * v_prev + (1 - beta2) * np.square(delta_layer)
        new_m.append(m_curr)
        new_v.append(v_curr)

    # Update global weights
    new_weights = []
    for idx, gw_layer in enumerate(global_weights):
        update = new_m[idx] / (np.sqrt(new_v[idx]) + eps)
        new_w = gw_layer + server_lr * update
        new_weights.append(new_w)

    return new_weights, new_m, new_v


In [ ]:
# Prepare datasets for clients (already fixed to avoid ImageDataGenerator slicing issues)
datagen = ImageDataGenerator(rescale=1./255)

full_train_gen = datagen.flow_from_directory(
    train_path,
    target_size=(img_height, img_width),
    batch_size=1,
    shuffle=False,
    class_mode='categorical'
)

X_train, y_train = [], []
for i in range(len(full_train_gen)):
    x, y = full_train_gen[i]
    X_train.append(x[0])
    y_train.append(y[0])

X_train = np.array(X_train)
y_train = np.array(y_train)

indices = np.arange(len(X_train))
np.random.shuffle(indices)
X_train = X_train[indices]
y_train = y_train[indices]

# Split among clients
client_train_gens = []
samples_per_client = len(X_train) // NUM_CLIENTS

for i in range(NUM_CLIENTS):
    start = i * samples_per_client
    end = (i + 1) * samples_per_client if i < NUM_CLIENTS - 1 else len(X_train)
    
    client_dataset = tf.data.Dataset.from_tensor_slices((X_train[start:end], y_train[start:end]))
    client_dataset = client_dataset.batch(batch_size)
    client_train_gens.append(client_dataset)


In [ ]:
#graph plotting for client and global model 

import matplotlib.pyplot as plt

# Track accuracies (put this before the training loop)
global_accuracies = []
client_accuracies_per_round = []

# Replace the inner loop in your federated training with tracking:
for round_num in range(NUM_ROUNDS):
    print(f"\n--- Round {round_num + 1} ---")
    client_models = []
    round_client_accuracies = []

    for client_index, client_gen in enumerate(client_train_generators):
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_gen, epochs=EPOCHS_PER_CLIENT, verbose=0)

        val_loss, val_acc = client_model.evaluate(client_val_generators[client_index], verbose=0)
        print(f'Client {client_index + 1} Validation Accuracy: {val_acc:.4f}')
        round_client_accuracies.append(val_acc)

        client_models.append(client_model)

    # Save client accuracies
    client_accuracies_per_round.append(round_client_accuracies)

    # Aggregate using FedAvg
    new_weights = []
    for layer_index in range(len(global_model.get_weights())):
        layer_weights = np.mean([client_models[i].get_weights()[layer_index] for i in range(NUM_CLIENTS)], axis=0)
        new_weights.append(layer_weights)
    global_model.set_weights(new_weights)

    # Evaluate global model
    val_loss, val_acc = global_model.evaluate(val_set, verbose=0)
    global_accuracies.append(val_acc)
    print(f'Global Model Validation Accuracy: {val_acc:.4f}')

# Plot Global Validation Accuracy over Rounds
plt.figure(figsize=(8, 5))
plt.plot(range(1, NUM_ROUNDS + 1), global_accuracies, marker='o', label='Global Validation Accuracy')
plt.title('Global Validation Accuracy Over Federated Rounds')
plt.xlabel('Federated Round')
plt.ylabel('Accuracy')
plt.xticks(range(1, NUM_ROUNDS + 1))
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()
# Plot Global Validation Accuracy over Rounds
plt.figure(figsize=(8, 5))
plt.plot(range(1, NUM_ROUNDS + 1), global_accuracies, marker='o', label='Global Validation Accuracy', color='blue')
plt.title('Global Validation Accuracy Over Federated Rounds')
plt.xlabel('Federated Round')
plt.ylabel('Accuracy')
plt.xticks(range(1, NUM_ROUNDS + 1))
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot Each Client's Accuracy Over Rounds
plt.figure(figsize=(10, 6))
for client_idx in range(NUM_CLIENTS):
    client_accs = [round_accs[client_idx] for round_accs in client_accuracies_per_round]
    plt.plot(range(1, NUM_ROUNDS + 1), client_accs, marker='o', label=f'Client {client_idx + 1}')

plt.title('Client Validation Accuracy Over Federated Rounds')
plt.xlabel('Federated Round')
plt.ylabel('Accuracy')
plt.xticks(range(1, NUM_ROUNDS + 1))
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


